<a href="https://colab.research.google.com/github/rontalapoojareddy/DeepLearning1/blob/main/Assignment_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**2503B05130**

**R.Pooja Reddy**

**M.Tech CSE**

In [ ]:
# =====================================================
# LoRA Fine-Tuning for Content Moderation
# =====================================================

# Install (run once)
# pip install transformers datasets peft accelerate scikit-learn

# =====================================================
# 1. IMPORT LIBRARIES
# =====================================================
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# =====================================================
# 2. DATA PREPARATION
# =====================================================
data = [
    {"text": "I hate you", "label": 1},
    {"text": "You are amazing", "label": 0},
    {"text": "This is stupid and awful", "label": 1},
    {"text": "Have a nice day", "label": 0},
    {"text": "You are dumb", "label": 1},
    {"text": "I love this!", "label": 0},
]

dataset = Dataset.from_list(data)

# Split dataset
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

# =====================================================
# 3. TOKENIZATION
# =====================================================
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True
    )

train_dataset = train_dataset.map(tokenize_function)
test_dataset = test_dataset.map(tokenize_function)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)
test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

# =====================================================
# 4. MODEL + LoRA
# =====================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin", "v_lin"],  # For DistilBERT
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, lora_config)

# =====================================================
# 5. METRICS
# =====================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary"
    )
    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

# =====================================================
# 6. TRAINING CONFIGURATION
# =====================================================
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    logging_dir="./logs"
)

# =====================================================
# 7. TRAINER SETUP
# =====================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# =====================================================
# 8. TRAIN MODEL
# =====================================================
trainer.train()

# =====================================================
# 9. EVALUATE MODEL
# =====================================================
evaluation_results = trainer.evaluate()
print("\nEvaluation Results:")
print(evaluation_results)

# =====================================================
# 10. SAVE MODEL
# =====================================================
model.save_pretrained("./lora_model")
tokenizer.save_pretrained("./lora_model")

# =====================================================
# 11. PREDICTION FUNCTION
# =====================================================
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    outputs = model(**inputs)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(probabilities).item()

    return {
        "text": text,
        "label": "Toxic" if predicted_class == 1 else "Safe",
        "confidence": float(probabilities[0][predicted_class])
    }

# =====================================================
# 12. TEST PREDICTIONS
# =====================================================
print("\nSample Predictions:")
print(predict("I hate this person"))
print(predict("You are awesome"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but n

Step,Training Loss


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/tmp/ipykernel_5607/85427964.py:162: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  "confidence": float(probabilities[0][predicted_class])



Evaluation Results:
{'eval_loss': 0.6814075708389282, 'eval_accuracy': 0.5, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1_score': 0.0, 'eval_runtime': 2.3561, 'eval_samples_per_second': 0.849, 'eval_steps_per_second': 0.424, 'epoch': 3.0}

Sample Predictions:
{'text': 'I hate this person', 'label': 'Safe', 'confidence': 0.5202297568321228}
{'text': 'You are awesome', 'label': 'Safe', 'confidence': 0.5321224927902222}


In [ ]:
# app.py

from fastapi import FastAPI
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

app = FastAPI()

model_path = "./lora_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

@app.get("/")
def home():
    return {"message": "Content Moderation API"}

@app.post("/moderate")
def moderate(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    label = torch.argmax(probs).item()

    return {
        "text": text,
        "label": "Toxic" if label == 1 else "Safe",
        "confidence": float(probs[0][label])
    }

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights: 0it [00:00, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: ./lora_model
Key                                                                                                    | Status     | 
-------------------------------------------------------------------------------------------------------+------------+-
base_model.model.distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.q_lin.lora_A.default.weight | UNEXPECTED | 
base_model.model.distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.q_lin.lora_B.default.weight | UNEXPECTED | 
base_model.model.distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.v_lin.lora_A.default.weight | UNEXPECTED | 
base_model.model.distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.v_lin.lora_B.default.weight | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.q_lin.lora_B.default.weight                  | MISSING    | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.v_lin.lora_B.default.weight               